# Statistical analysis of `gender_given` prompt cases

I want to be determine if there a behavioural difference across models when gender is specified to be one binary value or another. For this I am planning a logistic regression anaylsis to see if diversity and entropy metrics are informative of gender/have significant predictive power over the label.

## Load data

In [4]:
import sys
from pathlib import Path
root_dir = Path.cwd().parent
results_dir = root_dir / "data" / "olmo7b_results" / "v2"
sys.path.insert(0, str(root_dir))
from utils import load_json_data, save_dataframes

KEYWORD = "given"   # can otherwise be "assumed"
PREFIX = "combined"  # can be None

loaded_dfs = load_json_data(results_dir, 
                            file_name_keyword=KEYWORD, 
                            prefix=PREFIX)
print(f"Loaded the following {KEYWORD} dfs:")
for k in loaded_dfs.keys():
    print(k)

Loaded the following given dfs:
combined_metrics_olmo7b_given


In [5]:
list(loaded_dfs.values())[0].head()

,model_key,profile_id,temperature,occupation_category,attended_university,response_number,mean_entropy,mean_entropy_nucleus,gender,self_bleu,semantic_div,perplexity
0,base,1,0.2,officers in regular armed forces,no,1,0.206833,0.153161,male,0.34691,0.317608,11.525167
1,base,1,0.2,officers in regular armed forces,no,2,0.191472,0.143714,male,0.34691,0.317608,11.525167
2,base,1,0.2,officers in regular armed forces,no,3,0.154052,0.110273,male,0.34691,0.317608,11.525167
3,base,1,0.2,officers in regular armed forces,no,4,0.150479,0.100602,male,0.34691,0.317608,11.525167
4,base,1,0.2,officers in regular armed forces,no,5,0.136403,0.095563,male,0.34691,0.317608,11.525167


In [6]:
import pandas as pd
combined_df = None
if PREFIX == "combined":
    combined_df = list(loaded_dfs.values())[0]
    print("Combined df columns:", combined_df.columns)
    print("Combined df shape:", combined_df.shape)
elif len(loaded_dfs) > 1:
    # combine all dfs into one (vertically)
    all_dfs = list(loaded_dfs.values())
    combined_df = pd.concat(all_dfs).reset_index(drop=True)
    print("Combined df columns:", combined_df.columns)
    assert combined_df.shape[0] == sum(df.shape[0] for df in all_dfs), "Row count mismatch after concatenation"
    print("Combined df shape:", combined_df.shape)
    save_dataframes({f"combined_metrics_olmo7b_{KEYWORD}": combined_df}, results_dir)

# Print total entries per model
print(f"Entries per model:\n{combined_df['model_key'].value_counts()}")

Combined df columns: Index(['model_key', 'profile_id', 'temperature', 'occupation_category',
       'attended_university', 'response_number', 'mean_entropy',
       'mean_entropy_nucleus', 'gender', 'self_bleu', 'semantic_div',
       'perplexity'],
      dtype='str')
Combined df shape: (17600, 12)
Entries per model:
model_key
base    4400
dpo     4400
rlvr    4400
sft     4400
Name: count, dtype: int64


## Logistic Regression setup

Columns to encode = `[model_key, occupation_category, attended_university, gender]` and all except `occupation` can be *one-hot encoded*.
And the outcome variable would be a binary male/female (`gender`) variable. I will drop `response_number` and `profile_id` for the regression model.

Also need to group the entries by `profile_id` and average the entropy metrics (since the rest are already computed over groups).

In [54]:
# Encode gender to 0/1 for regression
combined_df['gender_code'] = combined_df['gender'].map({'female': 0, 'male': 1})

In [55]:
# Drop columns that are not necessary for regression
cols_to_drop = ['response_number', 'attended_university', 'occupation_category', 'gender', 'mean_entropy']
combined_df = combined_df.drop(columns=cols_to_drop, errors='ignore')
print("Columns after dropping unnecessary ones:", combined_df.columns)
# keeping temperature:controlling for the fact that higher temps produce higher entropy, is there still a gender signal in entropy?

Columns after dropping unnecessary ones: Index(['model_key', 'profile_id', 'temperature', 'mean_entropy_nucleus',
       'self_bleu', 'semantic_div', 'perplexity', 'gender_code'],
      dtype='object')


In [56]:
# Group by profile_id and average the entropy metrics
entropy_cols = ['mean_entropy_nucleus']#, 'mean_entropy_nucleus']
combined_df[entropy_cols] = combined_df.groupby('profile_id')[entropy_cols].transform('mean')
reg_df = combined_df.drop_duplicates(subset=['profile_id', 'model_key', 'temperature'])
print("Entropy metrics averaged by profile_id")
reg_df.head(10)

Entropy metrics averaged by profile_id


,model_key,profile_id,temperature,mean_entropy_nucleus,self_bleu,semantic_div,perplexity,gender_code
0,base,1,0.2,1.290958,0.346910,0.317608,11.525167,1
5,base,2,0.2,1.129948,0.549618,0.113976,13.220261,1
10,base,3,0.2,1.254101,0.493192,0.246709,10.853041,1
15,base,4,0.2,1.131646,0.455229,0.341614,11.104241,1
20,base,5,0.2,1.091030,0.393514,0.345137,12.437788,1
25,base,6,0.2,1.080948,0.332367,0.370686,13.732744,1
30,base,7,0.2,1.130778,0.455236,0.263575,14.642786,1
35,base,8,0.2,1.002463,0.469745,0.307749,12.405449,1
40,base,9,0.2,1.019025,0.444786,0.300871,12.549057,1
45,base,10,0.2,1.150734,0.427374,0.238464,12.521621,1


### Run logreg for each model subset

I will break down the df into 4 subsets: for each model, and run a regression model for each.

In [57]:
# Initialise scaler for all the continuous metric cols for regression
continuous_cols = ['mean_entropy_nucleus', 'semantic_div', 'self_bleu', 'perplexity']#, 'mean_entropy_nucleus']

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

# Scaling globally to keep information about the relative differences across models
reg_df[continuous_cols] = scaler.fit_transform(reg_df[continuous_cols])
reg_df.head(10)

C:\Users\manth\AppData\Local\Temp\ipykernel_28924\1527630044.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  reg_df[continuous_cols] = scaler.fit_transform(reg_df[continuous_cols])


,model_key,profile_id,temperature,mean_entropy_nucleus,self_bleu,semantic_div,perplexity,gender_code
0,base,1,0.2,2.963486,1.406959,-1.248415,-0.295328,1
5,base,2,0.2,0.704966,3.080691,-2.808065,-0.290362,1
10,base,3,0.2,2.446489,2.614793,-1.791442,-0.297297,1
15,base,4,0.2,0.728778,2.301339,-1.064546,-0.296561,1
20,base,5,0.2,0.159048,1.791762,-1.037558,-0.292654,1
25,base,6,0.2,0.017629,1.286883,-0.841875,-0.288860,1
30,base,7,0.2,0.716608,2.301399,-1.662261,-0.286194,1
35,base,8,0.2,-1.083298,2.421196,-1.323920,-0.292749,1
40,base,9,0.2,-0.850982,2.215109,-1.376604,-0.292328,1
45,base,10,0.2,0.996534,2.071341,-1.854585,-0.292408,1


Now I will run 4 regression models.

In [58]:
import statsmodels.formula.api as smf

formula = """
    gender_code ~ mean_entropy_nucleus  
               + self_bleu + semantic_div + perplexity
               + temperature  
"""

results = {}
for model_name, group in reg_df.groupby('model_key'):
    fit = smf.logit(formula, data=group).fit()
    results[model_name] = fit

# Then compare coefficients across models
for model_name, fit in results.items():
    print(f"\n=== {model_name} ===")
    print(fit.summary())

Optimization terminated successfully.
         Current function value: 0.691055
         Iterations 4
Optimization terminated successfully.
         Current function value: 0.689209
         Iterations 4
Optimization terminated successfully.
         Current function value: 0.689820
         Iterations 4
Optimization terminated successfully.
         Current function value: 0.690497
         Iterations 4

=== base ===
                           Logit Regression Results                           
Dep. Variable:            gender_code   No. Observations:                  880
Model:                          Logit   Df Residuals:                      874
Method:                           MLE   Df Model:                            5
Date:                Mon, 09 Mar 2026   Pseudo R-squ.:                0.003019
Time:                        20:37:09   Log-Likelihood:                -608.13
converged:                       True   LL-Null:                       -609.97
Covariance Type:         

## Regression analysis on 